# Customer Churn Analysis — Python EDA

This notebook validates the cleaned customer dataset and performs exploratory data analysis.

**Input:** `customer_churn_cleaned_fixed.csv`

**Important encoding:** `Churn = 1` means the customer churned; `Churn = 0` means the customer was retained. Binary fields such as `Partner`, `Dependents`, `SeniorCitizen`, and `PaperlessBilling` are also encoded as 0/1.

> The original raw dataset is not available in this project folder. Therefore, this notebook starts from the supplied cleaned dataset; it validates the data rather than claiming to repeat the original raw-data cleaning process.


In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
print("Customer Churn Analysis started!")

In [ ]:
# 2. Load the corrected cleaned dataset
possible_paths = [
    Path("Dataset/customer_churn_cleaned_fixed.csv"),  # Run from project root
    Path("../Dataset/customer_churn_cleaned_fixed.csv"),  # Run from Python folder
    Path("/content/customer_churn_cleaned_fixed.csv"),  # Google Colab Files panel
    Path("customer_churn_cleaned_fixed.csv")  # Same folder as notebook
]

csv_path = next((path for path in possible_paths if path.exists()), None)

if csv_path is None:
    raise FileNotFoundError(
        "customer_churn_cleaned_fixed.csv was not found. "
        "In Colab, upload it using the Files panel. "
        "For GitHub, place it in the Dataset folder."
    )

df = pd.read_csv(csv_path)
print("Loaded file:", csv_path)
df.head()

In [ ]:
# 3. Check dataset structure
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nDataset information:")
df.info()

In [ ]:
# 4. Validate expected columns
expected_columns = [
    "CustomerID", "Gender", "SeniorCitizen", "Partner", "Dependents",
    "TenureMonths", "ContractType", "InternetService", "PaymentMethod",
    "PaperlessBilling", "MonthlyCharges", "TotalCharges", "SupportTickets",
    "Churn", "SignupChannel", "SatisfactionScore", "TenureGroup"
]

missing_columns = sorted(set(expected_columns) - set(df.columns))
unexpected_columns = sorted(set(df.columns) - set(expected_columns))

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

if unexpected_columns:
    print("Additional columns found:", unexpected_columns)

print("Column validation passed.")
print("Number of columns:", len(df.columns))

In [ ]:
# 5. Check missing values and duplicate records
print("Missing values by column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate CustomerIDs:", df["CustomerID"].duplicated().sum())

In [ ]:
# 6. Check category distributions and numeric summaries
for column in ["Gender", "PaymentMethod", "ContractType", "InternetService",
               "SignupChannel", "Churn"]:
    print(f"\n{column} distribution:")
    print(df[column].value_counts(dropna=False))

print("\nMonthly Charges summary:")
display(df["MonthlyCharges"].describe())

print("\nSatisfaction Score distribution:")
print(df["SatisfactionScore"].value_counts().sort_index())

In [ ]:
# 7. Confirm data types and churn encoding
df["Churn"] = pd.to_numeric(df["Churn"], errors="raise")

if not set(df["Churn"].dropna().unique()).issubset({0, 1}):
    raise ValueError("Churn must contain only 0 and 1.")

print("Churn encoding verified: 1 = Churned, 0 = Retained")
print(df["Churn"].value_counts().sort_index())

In [ ]:
# 8. Overall churn and retention
total_customers = len(df)
churned_customers = int(df["Churn"].eq(1).sum())
retained_customers = int(df["Churn"].eq(0).sum())

churn_rate = (churned_customers / total_customers * 100) if total_customers else 0
retention_rate = (retained_customers / total_customers * 100) if total_customers else 0

print(f"Total customers: {total_customers:,}")
print(f"Churned customers: {churned_customers:,}")
print(f"Retained customers: {retained_customers:,}")
print(f"Churn rate: {churn_rate:.2f}%")
print(f"Retention rate: {retention_rate:.2f}%")

In [ ]:
# 9. Visualize overall churn distribution
churn_counts = df["Churn"].value_counts().reindex([1, 0], fill_value=0)

plt.figure(figsize=(7, 5))
bars = plt.bar(["Churned", "Retained"], churn_counts.values)
plt.title("Customer Churn Distribution")
plt.xlabel("Customer Status")
plt.ylabel("Number of Customers")

for bar, value in zip(bars, churn_counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 5,
             f"{value:,}", ha="center")

plt.tight_layout()
plt.show()

In [ ]:
# 10. Churn rate by contract type
contract_analysis = df.groupby("ContractType").agg(
    Total_Customers=("CustomerID", "count"),
    Churned_Customers=("Churn", lambda values: values.eq(1).sum())
)

contract_analysis["Churn_Rate"] = (
    contract_analysis["Churned_Customers"]
    / contract_analysis["Total_Customers"] * 100
).round(2)

contract_analysis = contract_analysis.sort_values("Churn_Rate", ascending=False)
display(contract_analysis)

In [ ]:
# 11. Visualize churn rate by contract type
plt.figure(figsize=(8, 5))
bars = plt.bar(contract_analysis.index, contract_analysis["Churn_Rate"])
plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate (%)")

for bar, value in zip(bars, contract_analysis["Churn_Rate"]):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.4,
             f"{value:.2f}%", ha="center")

plt.tight_layout()
plt.show()

In [ ]:
# 12. Validate tenure groups and analyze churn by tenure
tenure_labels = ["0-12 months", "13-24 months", "25-48 months", "49-72 months"]
expected_tenure_group = pd.cut(
    df["TenureMonths"],
    bins=[-1, 12, 24, 48, 72],
    labels=tenure_labels
).astype("string")

tenure_mismatch = df["TenureGroup"].astype("string").ne(expected_tenure_group)

print("Tenure group mismatches:", int(tenure_mismatch.sum()))

if tenure_mismatch.any():
    print("Warning: Some TenureGroup values do not match TenureMonths bins.")

tenure_analysis = df.groupby("TenureGroup", observed=False).agg(
    Total_Customers=("CustomerID", "count"),
    Churned_Customers=("Churn", lambda values: values.eq(1).sum())
)

tenure_analysis["Churn_Rate"] = (
    tenure_analysis["Churned_Customers"]
    / tenure_analysis["Total_Customers"].replace(0, np.nan) * 100
).round(2)

display(tenure_analysis)

In [ ]:
# 13. Churn rate by payment method
payment_analysis = df.groupby("PaymentMethod").agg(
    Total_Customers=("CustomerID", "count"),
    Churned_Customers=("Churn", lambda values: values.eq(1).sum())
)

payment_analysis["Churn_Rate"] = (
    payment_analysis["Churned_Customers"]
    / payment_analysis["Total_Customers"] * 100
).round(2)

payment_analysis = payment_analysis.sort_values("Churn_Rate", ascending=False)
display(payment_analysis)

In [ ]:
# 14. Churn rate by internet service
internet_analysis = df.groupby("InternetService").agg(
    Total_Customers=("CustomerID", "count"),
    Churned_Customers=("Churn", lambda values: values.eq(1).sum())
)

internet_analysis["Churn_Rate"] = (
    internet_analysis["Churned_Customers"]
    / internet_analysis["Total_Customers"] * 100
).round(2)

display(internet_analysis.sort_values("Churn_Rate", ascending=False))

In [ ]:
# 15. Correlation between selected numeric variables
numeric_columns = [
    "SeniorCitizen", "Partner", "Dependents", "TenureMonths",
    "PaperlessBilling", "MonthlyCharges", "TotalCharges",
    "SupportTickets", "Churn", "SatisfactionScore"
]

correlation = df[numeric_columns].corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap — Customer Churn Dataset")
plt.tight_layout()
plt.show()

In [ ]:
# 16. Final dataset validation and export
print("Final dataset shape:", df.shape)
print("Total missing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print(f"Final churn rate: {df['Churn'].eq(1).mean() * 100:.2f}%")

output_path = Path("customer_churn_cleaned_python_output.csv")
df.to_csv(output_path, index=False)
print("Exported dataset to:", output_path.resolve())

In [ ]:
# 17. Download the exported CSV when running in Google Colab
try:
    from google.colab import files
    files.download("customer_churn_cleaned_python_output.csv")
except ImportError:
    print("Not running in Google Colab. The CSV is saved in the current working folder.")